# Исследование рынка видеоигр

### Цели и задачи проекта

Цель проекта — изучить данные игровой индустрии за период с 2000 по 2013 год. В исходных данных, собранных из открытых источников, есть информация о продажах игр, жанрах и платформах, годах выпуска, оценках игр от критиков и пользователей, а также о возрастной категории.

Для достижения цели стоят следующие задачи: познакомиться с данными, проверить их корректность и провести предобработку, произвести срез данных по нужному периоду, провести категоризацию игр по оценкам пользователей и критиков, а также определить наиболее популярные платформы по количеству выпущенных на них игр (топ-7).

Игры разбиваются на три категории по оценкам:

- Высокая оценка: оценка пользователя от 8 до 10 и критика -  от 80 до 100 (включая верхнюю границу)
- Средняя оценка: оценка пользователя от 3 до 8 и критика - от 30 до 80, (не включая верхнюю границу)
- Низкая оценка: оценка пользователя  от 0 до 3 и критика - от 0 до 30, (не включая верхнюю границу).

### Описание данных

Данные /datasets/new_games.csv содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
- Name — название игры.
- Platform — название платформы.
- Year of Release — год выпуска игры.
- Genre — жанр игры.
- NA sales — продажи в Северной Америке (в миллионах проданных копий).
- EU sales — продажи в Европе (в миллионах проданных копий).
- JP sales — продажи в Японии (в миллионах проданных копий).
- Other sales — продажи в других странах (в миллионах проданных копий).
- Critic Score — оценка критиков (от 0 до 100).
- User Score — оценка пользователей (от 0 до 10).
- Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта

- 1) Загрузка данных и знакомство с ними
- 2) Проверка ошибок в данных и их предобработка (обработка названий столбцов, преобразование типов данных, обработка пропусков и дубликатов)
- 3) Фильтрация данных по нужным годам
- 4) Категоризация данных по оценкам, нахождение топ-7 платформ по кол-ву выпущенных на них игр
- 5) Итоги и выводы

---

## 1. Загрузка данных и знакомство с ними


Загрузим библиотеку pandas и данные датасета `/datasets/new_games.csv`.


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/datasets/new_games.csv')

Так выглядят данные (первые пять строк):

In [3]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Информация о данных:

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Пропуски:

In [5]:
df.isna().sum()

Name                  2
Platform              0
Year of Release     275
Genre                 2
NA sales              0
EU sales              0
JP sales              0
Other sales           0
Critic Score       8714
User Score         6804
Rating             6871
dtype: int64

In [6]:
length_df = len(df)
length_df

16956

Данные: 16956 записей, 11 столбцов. В 6 столбцах есть данные с пропусками. Названия столбцов отражают содержимое данных, но записаны в неудобном для работы виде. Также не все столбцы с правильными типами данных. 

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

Выведем названия всех столбцов датафрейма, проверим их стиль написания:


In [7]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

Приведем столбцы к стилю snake case:

In [8]:
df.columns = df.columns.str.replace(' ', '_').str.lower()

In [9]:
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

### 2.2. Типы данных


Проверим типы данных:

In [10]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

Типы количества продаж и рейтинга переведем в числовые

In [11]:
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')

In [12]:
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')

In [13]:
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

Типы данных после преобразования:

In [14]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating              object
dtype: object

Году выпуска (year_of_release) лучше подходит тип Int (но сначала нужно заполнить пустые значения)

### 2.3. Наличие пропусков в данных

Количество пропусков в каждом столбце (абсолютные значения):


In [15]:
df.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

Количество пропусков в каждом столбце (относительные значения):

In [16]:
df.isna().sum() / len(df)

name               0.000118
platform           0.000000
year_of_release    0.016218
genre              0.000118
na_sales           0.000000
eu_sales           0.000354
jp_sales           0.000236
other_sales        0.000000
critic_score       0.513918
user_score         0.546591
rating             0.405225
dtype: float64

Рассмотрим пропущенные значения.
- 2 пропуска в названии игры, скорее всего - ошибка. Удалим строки, их незначительное количество. У этих же строк пропущен и жанр
- 275 пропусков в столбце с годом выпуска игры. Возможно, в базах данных были пропуски в информации о выпуске малоизвестных игр. Заменим индикатором -1 (год выпуска не восстановить и ничем не заменить, а мы и так будем фильтровать с 2000 по 2013 год).
- Данные о продажах, возможно, не были собраны или были утеряны. Заменим на среднее значение в зависимости от названия платформы и года выхода игры
- Большой процент пропусков в данных об оценке критиков и пользователей, а также возрастной категории может возникнуть из-за необязательности сбора этой информации. Пропуски в рейтинге заполним значением "RP", т.к. не знаем присвоенный рейтинг. Для оценки пользователей и критиков посчитаем среднее, учитывая среднюю оценку жанра по платформе и году, чтобы не искажать данные.

Строки с пропущенным названием игры (у них и жанр пропущен):

In [17]:
df[df['name'].isnull()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14439,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


Удалим две строки с пропущенным названием игры:

In [18]:
df = df.dropna(subset=['name'])

Заполняем пустые значения года выпуска индикатором -1, а затем поменяем тип на Int

In [19]:
df['year_of_release'] = df['year_of_release'].fillna(-1.0)

In [20]:
df['year_of_release'] = pd.to_numeric(df['year_of_release'], downcast='integer')

Пропуски в данных с количеством проданных копий игры в том или ином регионе заменим на среднее значение в зависимости от названия платформы и года выхода игры

In [21]:
mean_eu = df.groupby(['year_of_release', 'platform'])['eu_sales'].transform('mean')
mean_jp = df.groupby(['year_of_release', 'platform'])['jp_sales'].transform('mean')

In [22]:
df['eu_sales'] = df['eu_sales'].fillna(mean_eu)
df['jp_sales'] = df['jp_sales'].fillna(mean_jp)

Пропуски в рейтинге заполним значением "RP", т.к. не знаем присвоенный рейтинг:

In [23]:
df['rating'] = df['rating'].fillna("RP")

Для оценки пользователей и критиков посчитаем среднее, учитывая среднюю оценку жанра по платформе и году

In [24]:
# среднее по жанру, году и платформе
mean_critic = df.groupby(['genre','year_of_release','platform'])['critic_score'].transform('mean')
mean_user = df.groupby(['genre','year_of_release','platform'])['user_score'].transform('mean')

# заполняем
df['critic_score'] = df['critic_score'].fillna(mean_critic)
df['user_score'] = df['user_score'].fillna(mean_user)

дозаполняем поэтапно, на случаи когда остаются пропуски по комбинации столбцов:

In [25]:
# среднее по жанру и платформе
p_g_mean_c = df.groupby(['genre', 'platform'])['critic_score'].transform('mean')
p_g_mean_u = df.groupby(['genre', 'platform'])['user_score'].transform('mean')

# заполняем
df['critic_score'] = df['critic_score'].fillna(p_g_mean_c)
df['user_score'] = df['user_score'].fillna(p_g_mean_u)

In [26]:
# дозаполняем средним
df['critic_score'] = df['critic_score'].fillna(df['critic_score'].mean())
df['user_score'] = df['user_score'].fillna(df['user_score'].mean())

Данные после обработки пропусков:

In [27]:
df.isna().sum()

name               0
platform           0
year_of_release    0
genre              0
na_sales           0
eu_sales           0
jp_sales           0
other_sales        0
critic_score       0
user_score         0
rating             0
dtype: int64

### 2.4. Явные и неявные дубликаты в данных

Изучим уникальные значения в категориальных данных: жанр игры, платформа, рейтинг и года выпуска. 

Видим, что возникли неявные дубликаты в названиях жанра из-за регистра:

In [28]:
df['genre'].unique()

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION', 'SHOOTER',
       'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE', 'SIMULATION',
       'PUZZLE', 'STRATEGY'], dtype=object)

Приведем к нижнему регистру:

In [29]:
df['genre'] = df['genre'].str.lower()

In [30]:
df['genre'].unique()

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy'], dtype=object)

Неявные дубликаты исчезли.

Дубликатов в рейтинге нет:

In [31]:
df['rating'].unique()

array(['E', 'RP', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC'], dtype=object)

Платформы уникальны:

In [32]:
df['platform'].unique()

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

Годы уникальны:

In [33]:
df['year_of_release'].unique()

array([2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010,
       2013, 2004, 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014,
       1992, 1997, 1993, 1994, 1982, 2016, 2003, 1986, 2000,   -1, 1995,
       1991, 1981, 1987, 1980, 1983], dtype=int16)

Проверим наличие явных дубликатов:

In [34]:
df.duplicated().sum()

212

Удалим совпадающие строки:

In [35]:
df.drop_duplicates(inplace=True)

Было найдено 212 дубликатов, они были удалены.

In [36]:
# проверим, что дубликатов не осталось
df.duplicated().sum()

0

Узнаем, как изменилась длина датасета:

In [37]:
# текущая длина датасета 
length_df_new = len(df)

In [38]:
# сколько строк были удалены
del_length = length_df - length_df_new
del_length

214

In [39]:
# процент удаленных строк
del_perc = round(100* del_length / length_df, 2)
del_perc

1.26

В процессе подготовки данных было удалено 214 строк (1.26%)

### Промежуточный вывод

Была произведена предобработка данных:
- изменен стиль написания столбцов
- преобразованы типы данных 
- произведена обработка пропусков
- устранены неявные дубликаты
- удалены дублирующиеся строки

В процессе подготовки данных было удалено 214 строк, что составило 1.26% от всех данных.

---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберем данные по этому показателю. Сохраним новый срез данных в отдельном датафрейме: `df_actual`.

In [40]:
df_actual = df.loc[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

Проверим, что фильтрация сработала и в новом датафрейме только нужные года:

In [41]:
df_actual['year_of_release'].unique()

array([2006, 2008, 2009, 2005, 2007, 2010, 2013, 2004, 2002, 2001, 2011,
       2012, 2003, 2000], dtype=int16)

In [42]:
df_actual.shape

(12804, 11)

После фильтрации данных осталось 12804 строки

---

## 4. Категоризация данных
    
Проведем категоризацию данных:
- Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [43]:
# проверим максимальное значение оценки
max(df_actual['user_score'])

9.7

In [44]:
# т.к. макс. значение <10, можем использовать cut с параметром right=False, не беспокоясь о граничном значении = 10
bins_u = [0, 3, 8, 10]
labels_u = ['низкая оценка', 'средняя оценка', 'высокая оценка']

df_actual['user_score_category'] = pd.cut(df_actual['user_score'], bins=bins_u, labels=labels_u, right=False)

- Разделим все игры по оценкам критиков и выделим такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [45]:
# проверим максимальное значение оценки
max(df_actual['critic_score'])

98.0

In [46]:
# т.к. макс. значение <100, можем использовать cut с параметром right=False,не беспокоясь о граничном значении = 100
bins_c = [0, 30, 80, 100]
labels_c = ['низкая оценка', 'средняя оценка', 'высокая оценка']

df_actual['critic_score_category'] = pd.cut(df_actual['critic_score'], bins=bins_c, labels=labels_c, right=False)

In [47]:
# посмотрим, как выглядит датасет с новыми полями
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,user_score_category,critic_score_category
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E,высокая оценка,средняя оценка
2,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E,высокая оценка,высокая оценка
3,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E,высокая оценка,высокая оценка
6,New Super Mario Bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89.0,8.5,E,высокая оценка,высокая оценка
7,Wii Play,Wii,2006,misc,13.96,9.18,2.93,2.84,58.0,6.6,E,средняя оценка,средняя оценка


- После категоризации данных проверяем результат: сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории:

In [48]:
df_actual.groupby('critic_score_category')['name'].count()


critic_score_category
низкая оценка        60
средняя оценка    10897
высокая оценка     1847
Name: name, dtype: int64

In [49]:
df_actual.groupby('user_score_category')['name'].count()

user_score_category
низкая оценка      131
средняя оценка    9584
высокая оценка    3089
Name: name, dtype: int64

- Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период:

In [50]:
df_actual['platform'].value_counts().head(7)

PS2     2128
DS      2125
Wii     1277
PSP     1182
X360    1122
PS3     1089
GBA      812
Name: platform, dtype: int64

---

## 5. Итоговый вывод


В рамках проекта были изучены данные игровой индустрии за период с 2000 по 2013 год. Проведена предобработка данных: изменены названия столбцов, скорректированы типы данных, обработаны пропуски, устранены неявные дубликаты и удалены дублирующиеся строки. На данном этапе длина датасета сократилась на 1.26%.

Был сформирован срез данных за 2000–2013 годы, содержащий 12804 строки. Для анализа добавлены два новых поля (категориальных признака): категории пользовательских и эеспертных оценок (низкая, средняя, высокая).

Также были определены наиболее популярные платформы по количеству выпущенных игр. Лидером является платформа PS2 — 2128 игр за рассматриваемый период.